# 🎵 MusicGen Fine-Tuning Journey
## Notebook 1: Setup & First Sounds

---

### What is this project?

We're going to **domain-adapt** a pre-trained music generation model to produce **Mongolian/Tuvan throat singing** — a style it has probably never been trained on. This is called *fine-tuning*, and it's one of the most powerful techniques in modern deep learning: take something that already knows a lot (MusicGen), and teach it something new (khoomei, sygyt, kargyraa).

Over 4 notebooks, we'll:
1. **[THIS ONE]** Understand the model and generate our first sounds
2. Collect and prepare a throat singing dataset
3. Fine-tune the model on our data
4. Evaluate and compare: base model vs. fine-tuned model

---

### The model we're using: MusicGen (Meta, 2023)

MusicGen is **autoregressive** — it generates music token-by-token, like a language model generates text word-by-word. Here's the flow:

```
Text prompt (e.g. "tuvan throat singing")
        │
        ▼
   T5 Text Encoder  ← frozen, converts text to embedding vectors
        │
        ▼
  Transformer LM   ← THIS IS WHAT WE FINE-TUNE
  (autoregressive)   predicts one audio token at a time
        │
        ▼
   EnCodec Decoder  ← frozen, converts tokens back to audio
        │
        ▼
   🔊 Audio waveform
```

**Key insight:** Audio is never processed as raw waveform by the LM. It goes through **EnCodec**, a learned audio codec that compresses audio into ~50 discrete tokens per second (vs. 32,000 raw samples per second). This makes it tractable for a transformer to model.

We freeze EnCodec (it's already great at encoding audio) and only train the transformer to produce better *sequences of tokens* for our new style.


## Step 1: Check Your Environment

The RTX 5090 uses NVIDIA's **Blackwell** architecture (compute capability 12.0). This requires:
- CUDA 12.8+
- PyTorch 2.7+

Run this cell first — it tells you exactly where you stand.

In [ ]:
import sys
import subprocess

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print(f"Python: {sys.version.split()[0]}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        cuda_version = torch.version.cuda
        compute_cap = torch.cuda.get_device_capability(0)

        print(f"\nGPU:          {gpu_name}")
        print(f"VRAM:         {vram_gb:.1f} GB")
        print(f"CUDA:         {cuda_version}")
        print(f"Compute cap:  {compute_cap[0]}.{compute_cap[1]}")

        # Blackwell check (5090 = GB202, compute 12.x)
        if compute_cap[0] >= 12:
            print("\n⚠️  Blackwell GPU detected (5090)")
            cuda_major = float(cuda_version.split('.')[0])
            torch_ver = tuple(int(x) for x in torch.__version__.split('.')[:2])
            if cuda_major >= 12 and torch_ver >= (2, 7):
                print("✅ PyTorch + CUDA versions look correct for Blackwell")
            else:
                print("❌ You may need: pip install torch>=2.7.0 --index-url https://download.pytorch.org/whl/cu128")
        else:
            print("\n✅ GPU detected (non-Blackwell, standard setup)")
    else:
        print("\n❌ No CUDA GPU found — check your WSL2 + NVIDIA driver setup")
        print("   Make sure you have 'nvidia-smi' working inside WSL2")
except ImportError:
    print("❌ PyTorch not installed. See requirements.txt for install instructions.")


## Step 2: Install AudioCraft + Visualization Tools

**AudioCraft** is Meta's open-source audio AI library — it includes MusicGen, AudioGen, and the training infrastructure.

> **Before running this:** Make sure you've installed PyTorch with the right CUDA version first (see `requirements.txt`).  
> Installing AudioCraft via pip will pull PyTorch as a dependency if it's missing, and it may pull the wrong CUDA version for your 5090.


In [ ]:
# Install AudioCraft and visualization libraries
# This may take 2-5 minutes the first time
!pip install audiocraft librosa soundfile matplotlib ipython --quiet
print("\n✅ Installation complete")

In [ ]:
# Verify all imports work
import torch
import torchaudio
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import IPython.display as ipd
from pathlib import Path

from audiocraft.models import MusicGen
from audiocraft.modules.conditioners import ConditioningAttributes

# Create output directory for this notebook
Path("nb1_outputs").mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ All imports OK — using device: {DEVICE}")

## Step 3: Load MusicGen-Small

We're using the **small** variant (300M parameters) for three reasons:
1. Fast enough to iterate on during development
2. Still large enough to generate coherent music
3. Easier to fine-tune — fewer parameters to update, less GPU memory needed

MusicGen also comes in `medium` (1.5B) and `large` (3.3B). Once you're comfortable with the pipeline, upgrading to medium is just changing one string.

The model will be downloaded from HuggingFace on first run (~1.2 GB).

In [ ]:
print("Loading MusicGen-small... (downloads ~1.2GB on first run)")
model = MusicGen.get_pretrained('facebook/musicgen-small', device=DEVICE)

# Count parameters in each component
lm_params = sum(p.numel() for p in model.lm.parameters())
encodec_params = sum(p.numel() for p in model.compression_model.parameters())

print(f"\n=== Model Architecture ===")
print(f"Language Model (Transformer):  {lm_params/1e6:.0f}M parameters  <- WE WILL FINE-TUNE THIS")
print(f"EnCodec (Audio Tokenizer):     {encodec_params/1e6:.0f}M parameters  <- stays frozen")
print(f"\nAudio sample rate:  {model.sample_rate} Hz")
print(f"Token rate:         ~{model.frame_rate} tokens/second per codebook")
print(f"Codebooks:          {model.compression_model.num_codebooks} (parallel streams of tokens)")

### Quick aside: What are codebooks?

EnCodec doesn't use a single stream of tokens — it uses **4 parallel codebooks**. Think of them as 4 layers of detail:

- **Codebook 0**: Coarse structure — overall pitch, rhythm, timbre
- **Codebook 1**: Medium detail — melodic contour, harmonic color  
- **Codebook 2**: Fine detail — texture, articulation
- **Codebook 3**: Very fine — subtle nuances, noise characteristics

Each codebook has a vocabulary of **2,048** possible tokens. So at each time step, you have 4 tokens, each from a set of 2,048 values. The LM predicts all 4 codebooks in a specific pattern (interleaved).

For 10 seconds of audio at ~50 tokens/second: **4 × 500 = 2,000 tokens total** per sample. That's what the transformer processes — not 320,000 raw audio samples.

## Step 4: Generate Your First Sounds

Before fine-tuning anything, let's see what MusicGen already knows. We'll ask it for throat singing and see what it produces — this gives us our **baseline** to compare against after fine-tuning.

**Generation parameters explained:**
- `duration`: length in seconds
- `top_k`: at each step, only sample from the top-K most likely tokens (controls diversity)
- `temperature`: > 1.0 = more random, < 1.0 = more conservative
- `cfg_coef`: classifier-free guidance — how strongly to follow the text (3.0–5.0 is typical)

In [ ]:
model.set_generation_params(
    duration=10,           # 10 seconds of audio
    use_sampling=True,     # sample from distribution (vs. greedy argmax)
    top_k=250,             # top-K sampling
    temperature=1.0,       # default temperature
    cfg_coef=3.0,          # text conditioning strength
)

# We're testing 4 prompts:
#   - Two throat singing prompts (our target domain)
#   - Two unrelated prompts (to confirm the model works generally)
descriptions = [
    "mongolian throat singing, khoomei style, meditative drone, ancient",
    "tuvan throat singing, sygyt, high overtone melody, traditional",
    "solo jazz piano, swing rhythm, bluesy, cafe",
    "electronic ambient, slow evolving pads, atmospheric",
]

print("Generating music... (~45-90 seconds)")
with torch.no_grad():
    wav = model.generate(descriptions)  # shape: [4, 1, samples]

print(f"\n✅ Done! Generated {wav.shape[0]} clips")
print(f"   Shape: {list(wav.shape)}  → [batch, channels, audio_samples]")
print(f"   Duration: {wav.shape[-1] / model.sample_rate:.1f}s each")

In [ ]:
# Save all generated clips and display them
sample_rate = model.sample_rate

for i, (desc, audio) in enumerate(zip(descriptions, wav)):
    audio_np = audio.squeeze(0).cpu().float().numpy()  # [samples]

    # Save to file
    filename = f"nb1_outputs/baseline_{i+1}.wav"
    sf.write(filename, audio_np, sample_rate)

    print(f"\n{'─'*60}")
    print(f"🎵 Prompt {i+1}: \"{desc}\"")
    print(f"   Saved: {filename}")
    display(ipd.Audio(audio_np, rate=sample_rate))

### 🤔 Reflection point #1

Listen to what MusicGen generates for the throat singing prompts. You'll probably notice:
- It might produce something vaguely "world music" or "eastern" sounding
- It almost certainly won't produce the actual overtone technique of khoomei/kargyraa
- The jazz and ambient prompts probably sound much more convincing

**Why?** MusicGen was trained predominantly on Western music from the internet. Tuvan throat singing is not well-represented in typical training data. This is exactly the gap we're closing with fine-tuning.

**Question to hold:** After fine-tuning, will the model produce better throat singing because it learned a new sound? Or will it get confused and lose quality on everything else?

## Step 5: Visualize — Mel Spectrograms

A **mel spectrogram** shows how energy is distributed across frequencies over time — it's the standard way to visually inspect audio in ML contexts.

- **X-axis:** Time
- **Y-axis:** Frequency (mel-scaled — compresses high frequencies like human hearing does)
- **Color:** Energy (brighter = louder at that frequency/time)

Throat singing has a very distinctive spectrogram: a strong low fundamental (the drone) plus sharp bright horizontal lines above it (the overtones/harmonics).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

for i, (desc, audio) in enumerate(zip(descriptions, wav)):
    audio_np = audio.squeeze(0).cpu().float().numpy()

    # Mel spectrogram
    S = librosa.feature.melspectrogram(
        y=audio_np,
        sr=sample_rate,
        n_mels=128,
        fmax=8000,
        n_fft=2048,
        hop_length=512
    )
    S_dB = librosa.power_to_db(S, ref=np.max)

    librosa.display.specshow(
        S_dB,
        x_axis='time',
        y_axis='mel',
        sr=sample_rate,
        ax=axes[i],
        fmax=8000
    )
    axes[i].set_title(f'"{desc[:50]}"', fontsize=9, wrap=True)
    axes[i].set_ylabel('Freq (Hz)')

plt.suptitle('Baseline Model — Mel Spectrograms', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('nb1_outputs/baseline_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: nb1_outputs/baseline_spectrograms.png")

## Step 6: Peek Under the Hood — Audio as Tokens

This is the most important conceptual step. Let's see what audio actually looks like to the language model — not as a waveform, but as **sequences of integers**.

This is what we'll be training the model to predict.

In [ ]:
# Take the first generated clip and encode it back to tokens
audio_sample = wav[0:1]  # shape: [1, 1, samples]

print("=" * 55)
print("WHAT AUDIO LOOKS LIKE TO THE LANGUAGE MODEL")
print("=" * 55)

print(f"\n[AS WAVEFORM]")
print(f"  Shape:     {list(audio_sample.shape)}")
print(f"  Samples:   {audio_sample.shape[-1]:,}")
print(f"  Rate:      {sample_rate} Hz")
print(f"  Duration:  {audio_sample.shape[-1]/sample_rate:.1f}s")

# Encode to discrete tokens using EnCodec
with torch.no_grad():
    codes, scale = model.compression_model.encode(audio_sample)

print(f"\n[AS TOKENS (what the LM sees)]")
print(f"  Shape:         {list(codes.shape)}   → [batch, codebooks, time_steps]")
print(f"  Codebooks:     {codes.shape[1]}  (parallel token streams)")
print(f"  Time steps:    {codes.shape[2]}  (at ~{codes.shape[2]/10:.0f} tokens/second)")
print(f"  Vocab size:    {model.compression_model.cardinality} possible values per token")
print(f"\n  Compression:   {audio_sample.shape[-1]:,} samples → {codes.shape[1]*codes.shape[2]:,} total tokens")
print(f"  Ratio:         {audio_sample.shape[-1]/(codes.shape[1]*codes.shape[2]):.0f}x compression")

print(f"\n[FIRST 30 TOKENS — Codebook 0 (coarsest)]")
print(codes[0, 0, :30].cpu().numpy())

print(f"\n[FIRST 30 TOKENS — Codebook 1]")
print(codes[0, 1, :30].cpu().numpy())

print(f"\n[FIRST 30 TOKENS — Codebook 2]")
print(codes[0, 2, :30].cpu().numpy())

print(f"\n[FIRST 30 TOKENS — Codebook 3 (finest)]")
print(codes[0, 3, :30].cpu().numpy())

print(f"\n💡 Fine-tuning teaches the LM which INTEGER SEQUENCES sound like throat singing.")
print(f"   The actual waveform never touches the LM — only these token IDs.")

In [ ]:
# Visualize the token patterns as a heatmap
fig, axes = plt.subplots(1, 4, figsize=(18, 3))

for k in range(4):
    cb_tokens = codes[0, k, :].cpu().numpy()  # [T]
    # Reshape into a grid for visualization
    T = len(cb_tokens)
    grid = cb_tokens.reshape(1, -1)
    im = axes[k].imshow(grid, aspect='auto', cmap='viridis', vmin=0, vmax=2047)
    axes[k].set_title(f'Codebook {k}', fontsize=10)
    axes[k].set_xlabel('Time step')
    axes[k].set_yticks([])
    plt.colorbar(im, ax=axes[k], label='Token ID')

plt.suptitle(f'Audio Tokens — "{descriptions[0][:40]}"', fontsize=11)
plt.tight_layout()
plt.savefig('nb1_outputs/token_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("Each column = one time step (1/50th of a second)")
print("Each color = a different token ID (0-2047)")
print("The LM learns to predict the next column given all previous columns + the text prompt.")

## Step 7: Verify the Round-Trip

EnCodec is a **codec** — it can both encode (audio → tokens) and decode (tokens → audio). Let's verify that the compression is good: encode our baseline audio then decode it, and compare to the original.

This also verifies our encode/decode pipeline before we use it in training.

In [ ]:
# Encode → Decode round trip
original_audio = wav[0:1]  # [1, 1, T]

with torch.no_grad():
    # Encode to tokens
    codes, scale = model.compression_model.encode(original_audio)

    # Decode back to audio
    reconstructed = model.compression_model.decode(codes, scale)  # [1, 1, T]

# Convert to numpy for comparison
orig_np = original_audio.squeeze().cpu().float().numpy()
recon_np = reconstructed.squeeze().cpu().float().numpy()

# Trim to same length (decode may add/remove a few samples)
min_len = min(len(orig_np), len(recon_np))
orig_np = orig_np[:min_len]
recon_np = recon_np[:min_len]

# Measure reconstruction quality
mse = np.mean((orig_np - recon_np) ** 2)
snr = 10 * np.log10(np.mean(orig_np**2) / (mse + 1e-10))

print(f"Round-trip reconstruction quality:")
print(f"  MSE: {mse:.6f}")
print(f"  SNR: {snr:.1f} dB  (>20dB = perceptually transparent for music)")

print("\n[Original]")
display(ipd.Audio(orig_np, rate=sample_rate))

print("[Reconstructed via tokens]")
display(ipd.Audio(recon_np, rate=sample_rate))

print("\n✅ The codec is working. Minor artifacts are expected — this is lossy compression.")
print("   The LM never needs to be 'perfect' — it just needs to predict these token sequences.")

## Notebook 1 Summary

**What we learned:**

1. MusicGen is a **two-part system**: EnCodec (audio↔tokens) + a Transformer LM (text+tokens→next token)
2. Audio is **tokenized to ~50 tokens/second** across 4 codebooks — 150x compression
3. The LM predicts **integer sequences**, not raw audio — same architecture as a language model
4. The **base model produces weak throat singing** — it's not in its training distribution
5. Fine-tuning = teaching the LM *which token sequences* correspond to throat singing

**Open questions going into Notebook 2:**
- How much data do we need? (Spoiler: maybe 30 minutes is enough to see a difference)
- What should our text descriptions look like? They matter more than you'd think
- Will a small model even be expressive enough to capture throat singing texture?

---

**➡️ Next: `02_data_preparation.ipynb` — Collecting and preparing your throat singing dataset**